<a href="https://colab.research.google.com/github/spectre729/cs_4782_final/blob/main/LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Load RoBERTa Model and Tokenizer and Datasets

In [ ]:
# Install necessary libraries
!pip install transformers datasets
!pip install -q --upgrade torchao

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, TaskType
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import gc

In [ ]:
from transformers import RobertaTokenizerFast, RobertaForSequenceClassification, AutoModelForSequenceClassification
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, TaskType

# Load pre-trained RoBERTa model and tokenizer
model_name = 'roberta-base'
tokenizer = RobertaTokenizerFast.from_pretrained(model_name)
model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

print(f"Loaded RoBERTa model: {model_name}")

### Load SST-2 Dataset and QNLI Dataset + Tokenize

In [ ]:
# Load SST-2 dataset
sst2_dataset = load_dataset('glue', 'sst2')
print("SST-2 dataset loaded successfully.")
display(sst2_dataset)

# Load QNLI dataset
qnli_dataset = load_dataset('glue', 'qnli')
print("QNLI dataset loaded successfully.")
display(qnli_dataset)

In [ ]:
def tokenize_sst2(example):
    return tokenizer(example['sentence'], truncation=True, padding='max_length', max_length=128)

def tokenize_qnli(example):
    return tokenizer(example['question'], example['sentence'], truncation=True, padding='max_length', max_length=128)

sst2_dataset = sst2_dataset.map(tokenize_sst2, batched=True)
sst2_dataset = sst2_dataset.rename_column("label", "labels")
sst2_dataset = sst2_dataset.remove_columns(["sentence", "idx"])

qnli_dataset = qnli_dataset.map(tokenize_qnli, batched=True)
qnli_dataset = qnli_dataset.rename_column("label", "labels")
qnli_dataset = qnli_dataset.remove_columns(["question", "sentence", "idx"])

#### LoRA
Let $W_0$ be the pre-trained weight matrix, then the low-rank update rule is $W_0' = W_0' + δW = W_0' + BA$

Modified forward pass: $h = W_0x + BAx$

Initialization: B is 0, A is random Gaussian distribution

Chosen model: RoBERTa

In [ ]:


import importlib, peft
importlib.reload(peft)
lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=16, # The paper suggests setting alpha = 2 * r
    target_modules=["query", "value"], # Paper's recommendation (Section 7.1)
    lora_dropout=0.1
)

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()



In [ ]:
from sklearn.metrics import accuracy_score
import numpy as np

# Define a function to compute metrics for sequence classification
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

training_args = TrainingArguments(
    learning_rate=5e-4,          # Specific LoRA LR from Paper Appendix D
    per_device_train_batch_size=32,
    num_train_epochs=5,         # GLUE tasks usually require more than 1 epoch
    fp16 = True,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_strategy="epoch",
    logging_steps= 100
    #fp16=True, # Enable mixed precision training
)

trainer = Trainer(
    model=lora_model,
    args=training_args,
    train_dataset=sst2_dataset['train'],
    eval_dataset=sst2_dataset['validation'],
    compute_metrics=compute_metrics
)
def extract_epoch_metrics(log_history):
    train_loss, val_loss, val_acc = [], [], []
    for entry in log_history:
        if "loss" in entry and "eval_loss" not in entry:
            train_loss.append(entry["loss"])
        if "eval_loss" in entry:
            val_loss.append(entry["eval_loss"])
        if "eval_accuracy" in entry:
            val_acc.append(entry["eval_accuracy"])
    return train_loss, val_loss, val_acc

trainer.train()
lora_train_loss, lora_val_loss, lora_val_acc = extract_epoch_metrics(trainer.state.log_history)
lora_results = {"train_loss": lora_train_loss, "val_loss": lora_val_loss, "val_acc": lora_val_acc}

print(f"Total Training Time: {trainer.state.log_history[-1]['train_runtime']/60:.2f} minutes")
# Merges Ba into W_original so the model runs at full speed
merged_model = lora_model.merge_and_unload()
merged_model.save_pretrained("./lora-final-model")


EPOCHS = 5
# Evaluation for Baseline RoBERTa
print("=== Evaluating Baseline RoBERTa ===")
# Reload the base model to ensure it's untainted for baseline evaluation
untrained_base_model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)
base_evaluation_trainer = Trainer(
    model=untrained_base_model, # Use the freshly loaded original base model
    args=training_args, # Reuse training args, primarily for eval_batch_size and output_dir
    eval_dataset=sst2_dataset['validation'],
    compute_metrics=compute_metrics,
)
base_eval_results = base_evaluation_trainer.evaluate()
print(f"Baseline RoBERTa Evaluation Results: {base_eval_results}")

# Evaluation for LoRA Fine-Tuned RoBERTa
print("\n=== Evaluating LoRA Fine-Tuned RoBERTa ===")
# The lora_model was trained and then merged into merged_model. Use merged_model for LoRA evaluation.
lora_evaluation_trainer = Trainer(
    model=merged_model, # Use the LoRA fine-tuned and merged model
    args=training_args, # Reuse training args, primarily for eval_batch_size and output_dir
    eval_dataset=sst2_dataset['validation'],
    compute_metrics=compute_metrics,
)
lora_eval_results = lora_evaluation_trainer.evaluate()
print(f"LoRA Fine-Tuned RoBERTa Evaluation Results: {lora_eval_results}")

# Clean up some memory
del base_evaluation_trainer
del lora_evaluation_trainer
del untrained_base_model # Clean up the reloaded base model too
torch.cuda.empty_cache()
gc.collect()

In [ ]:
from sklearn.metrics import accuracy_score
ft_model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, predictions)}

ft_training_args = TrainingArguments(
    output_dir="./ft_output",
    learning_rate=5e-4,           # standard full FT lr from paper
    per_device_train_batch_size=32,
    num_train_epochs=5,
    # fp16=True,
    weight_decay=0.01,
    save_strategy="epoch",
    eval_strategy="epoch",
    logging_strategy="epoch",
)

ft_trainer = Trainer(
    model=ft_model,
    args=ft_training_args,
    train_dataset=sst2_dataset['train'],
    eval_dataset=sst2_dataset['validation'],
    compute_metrics=compute_metrics,
)

ft_trainer.train()
print(f"Full FT Training Time: {ft_trainer.state.log_history[-1]['train_runtime']/60:.2f} minutes")

ft_train_loss, ft_val_loss, ft_val_acc = extract_epoch_metrics(ft_trainer.state.log_history)
ft_results = {"train_loss": ft_train_loss, "val_loss": ft_val_loss, "val_acc": ft_val_acc}

del ft_model, ft_trainer
torch.cuda.empty_cache()
gc.collect()

In [ ]:
x = [i + 1 for i in range(EPOCHS)]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ----- Create correct x-axes -----
x_lora_train = range(1, len(lora_results["train_loss"]) + 1)
x_lora_val   = range(1, len(lora_results["val_loss"]) + 1)

x_ft_train = range(1, len(ft_results["train_loss"]) + 1)
x_ft_val   = range(1, len(ft_results["val_loss"]) + 1)

# ---------------- Loss Plot ----------------
ax1.plot(x_lora_train,
         lora_results["train_loss"],
         marker='o',
         label='LoRA Train Loss')

ax1.plot(x_lora_val,
         lora_results["val_loss"],
         marker='o',
         label='LoRA Val Loss')

ax1.plot(x_ft_train,
         ft_results["train_loss"],
         marker='s',
         linestyle='--',
         label='Full FT Train Loss')

ax1.plot(x_ft_val,
         ft_results["val_loss"],
         marker='s',
         linestyle='--',
         label='Full FT Val Loss')

ax1.set_title("LoRA vs Full Fine-Tuning Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

# ---------------- Accuracy Plot ----------------
ax2.plot(x_lora_val,
         lora_results["val_acc"],
         marker='o',
         label='LoRA Val Accuracy')

ax2.plot(x_ft_val,
         ft_results["val_acc"],
         marker='s',
         linestyle='--',
         label='Full FT Val Accuracy')

ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

In [ ]:
### Experiment 1: Examining the effect of various rank

# the paper claims that a low rank is sufficient for LoRA. We will put that to the test
from sklearn.metrics import accuracy_score

def run_experiment():
  """
  Create a new LoRA model with some parameters and run the experiment.
  :)
  """

  rank_lst = [1, 2, 6, 16, 32]
  base_result = []
  lora_result = []

  for rank in rank_lst:

    model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)

    lora_config = LoraConfig(
      task_type=TaskType.SEQ_CLS,
      r=rank,
      lora_alpha=16, # The paper suggests setting alpha = 2 * r
      target_modules=["query", "value"], # Paper's recommendation (Section 7.1)
      lora_dropout=0.1)

    lora_model = get_peft_model(model, lora_config)
    training_args = TrainingArguments(
      output_dir=f"./rank_{rank}_output",
      learning_rate=5e-4,          # Specific LoRA LR from Paper Appendix D
      per_device_train_batch_size=32,
      num_train_epochs=3,         # GLUE tasks usually require more than 1 epoch
      fp16 = True,
      weight_decay=0.01,
      save_strategy="epoch",
      logging_strategy="epoch",
      logging_steps= 100
      #fp16=True, # Enable mixed precision training
    )

    trainer = Trainer(
        model=lora_model,
        args=training_args,
        train_dataset=sst2_dataset['train']
        # compute_metrics=compute_metrics
    )

    trainer.train()

    print(f"Total Training Time: {trainer.state.log_history[-1]['train_runtime']/60:.2f} minutes")
    # Merges Ba into W_original so the model runs at full speed
    def compute_metrics(eval_pred):
      logits, labels = eval_pred
      predictions = np.argmax(logits, axis=-1)
      return {"accuracy": accuracy_score(labels, predictions)}


    print(f"Evaluating Performance for Rank: {rank}")
    # Evaluation for Baseline RoBERTa
    print("=== Evaluating Baseline RoBERTa ===")
    # Reload the base model to ensure it's untainted for baseline evaluation
    untrained_base_model = RobertaForSequenceClassification.from_pretrained(model_name, num_labels=2)
    base_evaluation_trainer = Trainer(
        model=untrained_base_model, # Use the freshly loaded original base model
        args=training_args, # Reuse training args, primarily for eval_batch_size and output_dir
        eval_dataset=sst2_dataset['validation'],
        compute_metrics=compute_metrics,
    )
    base_results = base_evaluation_trainer.evaluate()
    print(f"Baseline RoBERTa Evaluation Results: {base_results}")

    # Evaluation for LoRA Fine-Tuned RoBERTa
    print("\n=== Evaluating LoRA Fine-Tuned RoBERTa ===")
    # The lora_model was trained and then merged into merged_model. Use merged_model for LoRA evaluation.
    lora_evaluation_trainer = Trainer(
        model=lora_model, # Use the LoRA fine-tuned and merged model
        args=training_args, # Reuse training args, primarily for eval_batch_size and output_dir
        eval_dataset=sst2_dataset['validation'],
        compute_metrics=compute_metrics,
    )
    lora_results = lora_evaluation_trainer.evaluate()
    print(f"LoRA Fine-Tuned RoBERTa Evaluation Results: {lora_results}")
    base_result.append(base_results['eval_accuracy'])
    lora_result.append(lora_results['eval_accuracy'])
    del model, lora_model, lora_config, trainer
    del untrained_base_model, base_evaluation_trainer, lora_evaluation_trainer
    torch.cuda.empty_cache()
    gc.collect()
  return rank_lst, base_result, lora_result

_, base, lora = run_experiment()
print("DONE")

In [ ]:

print(f"Base list: {base}") # a bit pointless
print(f"Lora list: {lora}")

# very similar, may have to run for 16 and 32

In [ ]:
# Clean up some memory
# del base_evaluation_trainer
# del lora_evaluation_trainer
torch.cuda.empty_cache()
gc.collect()

In [ ]:
output_dir = "./lora_roberta_saved"
lora_model.save_pretrained(output_dir)
print(f"LoRA model saved to {output_dir}")

## LoRA Re-implementation

Re-implementing Table 2 (RoBERTa-base on GLUE)

Using the authors' own loralib package directly (not HuggingFace PEFT).


In [ ]:
# Install the authors' own loralib package + dependencies
!pip install -q loralib transformers datasets scikit-learn

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import loralib as lora
from sklearn.metrics import accuracy_score
from transformers import (
    RobertaTokenizerFast,
    RobertaForSequenceClassification,
    TrainingArguments,
    Trainer,
)
from transformers.models.roberta.modeling_roberta import RobertaSelfAttention
from datasets import load_dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

## 1. LoRA Layer Injection

The paper's key idea: for a frozen weight matrix $W_0 \in \mathbb{R}^{d \times k}$, the adapted forward pass becomes:

$$h = W_0 x + \Delta W x = W_0 x + BAx$$

where $B \in \mathbb{R}^{d \times r}$, $A \in \mathbb{R}^{r \times k}$, and $r \ll \min(d, k)$.


In [ ]:
def apply_lora(model, r=8, lora_alpha=8, lora_dropout=0.0, target_modules=("query", "value")):
    """
    Replace specified attention projections with lora.Linear in every self-attention layer.
    Copies pretrained weights over so we don't lose them.
    Freezes all non-LoRA parameters via lora.mark_only_lora_as_trainable.
    """
    for module in model.modules():
        if not isinstance(module, RobertaSelfAttention):
            continue

        hidden   = module.query.in_features
        head_dim = module.query.out_features

        if "query" in target_modules:
            new_q = lora.Linear(hidden, head_dim, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)
            new_q.weight.data = module.query.weight.data.clone()  # copy pretrained W
            new_q.bias.data   = module.query.bias.data.clone()
            module.query = new_q

        if "value" in target_modules:
            new_v = lora.Linear(hidden, head_dim, r=r, lora_alpha=lora_alpha, lora_dropout=lora_dropout)
            new_v.weight.data = module.value.weight.data.clone()
            new_v.bias.data   = module.value.bias.data.clone()
            module.value = new_v

    # Freeze everything except parameters named "lora_*"
    lora.mark_only_lora_as_trainable(model)
    return model


def count_params(model):
    """Return (trainable, total) parameter counts."""
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    return trainable, total

## 2. Load and Tokenize Datasets

In [ ]:
TOKENIZER = RobertaTokenizerFast.from_pretrained("roberta-base")

def load_glue(task, max_length=128):
    """Download a GLUE task, tokenize, and return as a DatasetDict of tensors."""
    dataset = load_dataset("glue", task)

    if task == "sst2":
        tokenize     = lambda ex: TOKENIZER(ex["sentence"], truncation=True, padding="max_length", max_length=max_length)
        drop_columns = ["sentence", "idx"]
    elif task == "qnli":
        # QNLI encodes a (question, sentence) pair
        tokenize     = lambda ex: TOKENIZER(ex["question"], ex["sentence"], truncation=True, padding="max_length", max_length=max_length)
        drop_columns = ["question", "sentence", "idx"]

    dataset = dataset.map(tokenize, batched=True)
    dataset = dataset.rename_column("label", "labels")
    dataset = dataset.remove_columns(drop_columns)
    dataset.set_format("torch")
    return dataset


sst2 = load_glue("sst2")
qnli = load_glue("qnli")

print("SST-2:", sst2)
print("QNLI: ", qnli)

## 3. Training Helper

Default to 10 epochs in train_and_eval() function but the paper used 60.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds)}


def train_and_eval(model, train_data, eval_data, output_dir,
                   lr=5e-4, epochs=10, batch_size=16):
    args = TrainingArguments(
        output_dir=output_dir,
        learning_rate=lr,
        per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=64,
        num_train_epochs=epochs,
        eval_strategy="epoch",
        logging_strategy="epoch",
        save_strategy="no",
        fp16=torch.cuda.is_available(),
        weight_decay=0.01,
        warmup_ratio=0.06,
        lr_scheduler_type="linear",
        logging_steps=200,
        report_to="none",
    )
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_data,
        eval_dataset=eval_data,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    final_metrics = trainer.evaluate()
    logs = trainer.state.log_history

    train_loss = []
    val_loss = []
    val_acc = []

    for log in logs:
        if "loss" in log and "eval_loss" not in log:
            train_loss.append(log["loss"])

        if "eval_loss" in log:
            val_loss.append(log["eval_loss"])

        if "eval_accuracy" in log:
            val_acc.append(log["eval_accuracy"])

    return {
        "final_metrics": final_metrics,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_acc": val_acc,
    }

In [ ]:
lora_results = train_and_eval(
    model_lora,
    sst2["train"],
    sst2["validation"],
    output_dir="./out/lora_sst2",
    lr=5e-4,
    epochs=EPOCHS
)

ft_results = train_and_eval(
    model_ft,
    sst2["train"],
    sst2["validation"],
    output_dir="./out/ft_sst2",
    lr=2e-5,
    epochs=EPOCHS
)

In [ ]:
x = [i + 1 for i in range(EPOCHS)]
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# ----- Create correct x-axes -----
x_lora_train = range(1, len(lora_results["train_loss"]) + 1)
x_lora_val   = range(1, len(lora_results["val_loss"]) + 1)

x_ft_train = range(1, len(ft_results["train_loss"]) + 1)
x_ft_val   = range(1, len(ft_results["val_loss"]) + 1)

# ---------------- Loss Plot ----------------
ax1.plot(x_lora_train,
         lora_results["train_loss"],
         marker='o',
         label='LoRA Train Loss')

ax1.plot(x_lora_val,
         lora_results["val_loss"],
         marker='o',
         label='LoRA Val Loss')

ax1.plot(x_ft_train,
         ft_results["train_loss"],
         marker='s',
         linestyle='--',
         label='Full FT Train Loss')

ax1.plot(x_ft_val,
         ft_results["val_loss"],
         marker='s',
         linestyle='--',
         label='Full FT Val Loss')

ax1.set_title("LoRA vs Full Fine-Tuning Loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.legend()

# ---------------- Accuracy Plot ----------------
ax2.plot(x_lora_val,
         lora_results["val_acc"],
         marker='o',
         label='LoRA Val Accuracy')

ax2.plot(x_ft_val,
         ft_results["val_acc"],
         marker='s',
         linestyle='--',
         label='Full FT Val Accuracy')

ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.legend()

plt.tight_layout()
plt.show()

Full fine-tuning achieved slightly higher validation accuracy, but exhibited significant overfitting as shown by rapidly increasing validation loss. In contrast, LoRA maintained relatively stable validation loss while achieving competitive accuracy, demonstrating improved training stability and parameter efficiency.